# מטלה 1 – איסוף נתוני סרטים

מקורות: IMDb Non-Commercial Datasets + Wikipedia scraping  
טווח: שמות סרטים המתחילים ב-**Bm עד Bz**

> ה-scraping מ-Wikipedia מבוצע עם **ThreadPoolExecutor** (5 workers במקביל)  
> לצורך קיצור זמן הריצה משמעותית תוך שמירה על נימוס מול השרת.

# פרטי הגשה

| | |
|---|---|
||

**קישור למאגר:** [https://github.com/reuvenkazorer27/python_assignment](https://github.com/reuvenkazorer27/python_assignment)

In [ ]:
# ייבוא ספריות נדרשות
import pandas as pd           # עיבוד טבלאות נתונים
import requests               # שליחת בקשות HTTP
import re                     # ביטויים רגולריים לניתוח טקסט
import json                   # קריאה/כתיבה של קבצי cache
import time                   # השהיה בין בקשות לשרת
import threading              # סנכרון גישה לcache בין threads
from concurrent.futures import ThreadPoolExecutor, as_completed  # ריצת threads במקביל
from pathlib import Path      # ניהול נתיבי קבצים
from bs4 import BeautifulSoup # פענוח HTML לscraping

# תיקיית הנתונים של IMDb
DATA_DIR = Path('imdb_data')
DATA_DIR.mkdir(exist_ok=True)

# טווח אותיות לסינון שמות הסרטים
PREFIX_START = 'bm'
PREFIX_END   = 'bz'

## 1. הורדת קבצי IMDb

In [ ]:
# קבצי IMDb שנדרשים לפרויקט
IMDB_FILES = [
    'title.basics.tsv.gz',      # פרטי כותרות (שם, שנה, סוג, אורך, ז'אנר)
    'title.ratings.tsv.gz',     # דירוגי משתמשים
    'title.principals.tsv.gz',  # שחקנים וצוות לפי סרט
    'name.basics.tsv.gz',       # פרטי אנשים (שם, תפקידים)
]

# הורדה אוטומטית בריצה ראשונה — ריצות עוקבות משתמשות בקבצים הקיימים
for filename in IMDB_FILES:
    path = DATA_DIR / filename
    if path.exists():
        print('קיים:', filename)
        continue
    url = 'https://datasets.imdbws.com/' + filename
    print('מוריד:', filename)
    r = requests.get(url, stream=True, timeout=180)
    with open(path, 'wb') as f:
        for chunk in r.iter_content(512 * 1024):
            f.write(chunk)
    print('הורד:', filename)

print('הורדה הסתיימה.')

## 2. טעינה וסינון

In [ ]:
# קריאת טבלת הבסיס של IMDb — רק עמודות נדרשות לחיסכון בזיכרון
basics = pd.read_csv(
    DATA_DIR / 'title.basics.tsv.gz',
    sep='\t', na_values='\\N',
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres'],
    low_memory=False
)

# פילטר 1: סרטים בלבד (ללא סדרות, קצרות מטרז' וכד')
basics = basics[basics['titleType'] == 'movie'].copy()
print('סרטים:', len(basics))

# פילטר 2: שנות הפקה עד 2024 כולל
basics['startYear'] = pd.to_numeric(basics['startYear'], errors='coerce')
basics = basics[basics['startYear'] <= 2024]
print('אחרי סינון שנה:', len(basics))

# פילטר 3: אורך סרט בין 60 ל-300 דקות (הסרת קצרצרים וחריגים)
basics['runtimeMinutes'] = pd.to_numeric(basics['runtimeMinutes'], errors='coerce')
basics = basics[(basics['runtimeMinutes'] >= 60) & (basics['runtimeMinutes'] <= 300)]
print('אחרי סינון אורך:', len(basics))

# פילטר 4: שמות סרטים המתחילים באותיות Bm עד Bz (בהתאם להקצאת הקבוצה)
prefix = basics['primaryTitle'].str.lower().str[:2]
basics = basics[(prefix >= PREFIX_START) & (prefix <= PREFIX_END)].copy()
print('אחרי סינון אותיות:', len(basics))

# המרת שדה הז'אנרים ממחרוזת לרשימה
basics['genres'] = basics['genres'].apply(lambda x: x.split(',') if isinstance(x, str) else [])

basics[['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']].head()

## 3. מיזוג עם דירוגים

In [ ]:
# LEFT JOIN עם טבלת הדירוגים — שומר את כל הסרטים גם ללא דירוג
ratings = pd.read_csv(DATA_DIR / 'title.ratings.tsv.gz', sep='\t', na_values='\\N')

movies = basics.merge(ratings, on='tconst', how='left')
print('סרטים לאחר מיזוג דירוגים:', len(movies))

movies[['tconst', 'primaryTitle', 'averageRating', 'numVotes']].head()

## 4. שחקנים מובילים

In [ ]:
# קריאת טבלת השחקנים והצוות
principals = pd.read_csv(
    DATA_DIR / 'title.principals.tsv.gz',
    sep='\t', na_values='\\N',
    usecols=['tconst', 'ordering', 'nconst', 'category']
)

# סינון: שחקנים בלבד, רק לסרטים שנכנסו לסט
movie_ids = set(movies['tconst'])
actors = principals[
    (principals['category'] == 'actor') &
    (principals['tconst'].isin(movie_ids))
].copy()

# מיון לפי ordering (סדר קרדיט) ולקיחת עד 5 שחקנים ראשונים לכל סרט
actors = actors.sort_values(['tconst', 'ordering'])
lead = (
    actors.groupby('tconst')
    .head(5)
    .groupby('tconst')['nconst']
    .apply(list)
    .reset_index()
    .rename(columns={'nconst': 'lead_actors_ids'})
)

# LEFT JOIN — סרטים ללא שחקנים מקבלים רשימה ריקה
movies = movies.merge(lead, on='tconst', how='left')
movies['lead_actors_ids'] = movies['lead_actors_ids'].apply(lambda x: x if isinstance(x, list) else [])
print('לאחר הוספת שחקנים:', len(movies))

movies[['tconst', 'primaryTitle', 'lead_actors_ids']].head(3)

## 5. Wikipedia – שפה, מדינה, תקציב, הכנסות, עלילה (Threading)

In [ ]:
# קבצי cache לשמירת תוצאות הscraping בין ריצות
WIKI_CACHE_FILE = Path('wiki_cache.json')   # סרטים שנמצאו עם נתונים
WIKI_MISS_FILE  = Path('wiki_miss.json')    # סרטים שאושר שאינם בויקיפדיה
WIKI_BASE = 'https://en.wikipedia.org'
HEADERS   = {'User-Agent': 'MovieDataCollector/1.0 (course-assignment)'}

# מנעול לסנכרון גישה לcache בין threads שרצים במקביל
cache_lock = threading.Lock()

# טעינת cache ראשי — מסנן רשומות ריקות (כשלונות ריצה קודמת)
if WIKI_CACHE_FILE.exists():
    with open(WIKI_CACHE_FILE, encoding='utf-8') as f:
        _raw = json.load(f)
    cache = {k: v for k, v in _raw.items()
             if any(x is not None for x in v.values())}
    print(f'Hit-cache:  {len(cache)} entries with data')
else:
    cache = {}
    print('Hit-cache:  empty')

# טעינת cache החסרים — dconsts שאושר שאין להם דף ויקיפדיה
if WIKI_MISS_FILE.exists():
    with open(WIKI_MISS_FILE, encoding='utf-8') as f:
        miss_cache = set(json.load(f))
    print(f'Miss-cache: {len(miss_cache)} confirmed-not-found entries')
else:
    miss_cache = set()
    print('Miss-cache: empty')


# --- פונקציית המרת תקציב/הכנסות ממחרוזת למיליוני דולר ---

def parse_money(text):
    if not text:
        return None
    # הסרת סוגריים, הערות ותווים מיוחדים
    text = re.sub(r'[\[\]()*]', '', text)
    # נרמול מקפים וגרשיים (en-dash, em-dash, non-breaking space)
    text = text.replace('–', '-').replace('—', '-').replace(' ', ' ')
    text = text.strip()

    # מיליארדים
    m = re.search(r'\$?\s*([\d,.]+)\s*billion', text, re.I)
    if m:
        return round(float(m.group(1).replace(',', '')) * 1000, 2)

    # טווח: $53-72 million או $5.7-6.5 million → ממוצע
    m = re.search(r'\$?\s*([\d,.]+)\s*-\s*([\d,.]+)\s*million', text, re.I)
    if m:
        lo = float(m.group(1).replace(',', ''))
        hi = float(m.group(2).replace(',', ''))
        return round((lo + hi) / 2, 2)

    # מיליונים ישיר
    m = re.search(r'\$?\s*([\d,.]+)\s*million', text, re.I)
    if m:
        return round(float(m.group(1).replace(',', '')), 2)

    # סכום גולמי בדולרים (ממיר למיליונים אם מעל מיליון)
    m = re.search(r'\$\s*([\d,]+)', text)
    if m:
        val = float(m.group(1).replace(',', ''))
        if val >= 1_000_000:
            return round(val / 1_000_000, 2)
    return None


# --- שליחת בקשת HTTP עם טיפול בשגיאות ---

def _fetch(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15, allow_redirects=True)
        if r.status_code == 200:
            return r.text
    except Exception:
        pass
    return None


def _title_to_url(title):
    # המרת שם סרט לכתובת ויקיפדיה תקנית
    return WIKI_BASE + '/wiki/' + requests.utils.quote(title.replace(' ', '_'), safe='_()')


# --- זיהוי עמוד disambiguation (סרטים בשם זהה) ---

def _is_disambiguation(soup):
    return bool(soup.find('div', id='disambigbox')) or bool(
        soup.find('div', class_=re.compile(r'dmbox-disambig')))


def _follow_disambiguation(soup, year_s):
    # מחפש קישור לסרט המתאים לפי שנה בתוך עמוד disambiguation
    film_links = soup.find_all('a', href=re.compile(r'/wiki/.*[Ff]ilm'))
    if year_s:
        for a in film_links:
            if year_s in a.get('href', ''):
                return WIKI_BASE + a['href']
    if film_links:
        return WIKI_BASE + film_links[0]['href']
    for a in soup.find_all('a', href=re.compile(r'/wiki/')):
        txt = a.get_text(strip=True)
        if year_s and year_s in txt and 'film' in txt.lower():
            return WIKI_BASE + a['href']
    return None


# --- בדיקת תקינות עמוד — האם מדובר בדף סרט אמיתי ---

def _is_film_page(soup, year_s):
    if _is_disambiguation(soup):
        return False, None
    infobox = soup.find('table', class_=re.compile(r'infobox'))
    if not infobox:
        return False, None
    text = infobox.get_text(' ', strip=True).lower()
    # בדיקה שה-infobox מכיל שדות אופייניים לסרט
    if 'directed by' not in text and 'screenplay' not in text and 'film' not in text:
        return False, infobox
    # בדיקת שנה — פסילה אם הפרש יותר מ-2 שנים
    if year_s:
        m = re.search(r'(19|20)\d{2}', text)
        if m and abs(int(m.group(0)) - int(year_s)) > 2:
            return False, infobox
    return True, infobox


# --- חילוץ שדות מה-infobox של הסרט ---

def _extract_from_infobox(infobox):
    info = {'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None}
    for row in infobox.find_all('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        key = th.get_text(' ', strip=True).lower()
        # הסרת הערות שוליים לפני חילוץ הערך
        for sup in td.find_all('sup'):
            sup.decompose()
        val = td.get_text(' ', strip=True)
        if 'language' in key and info['Language'] is None:
            info['Language'] = val.split('\n')[0].strip() or None
        elif 'countr' in key and info['Country'] is None:
            info['Country'] = val.split('\n')[0].strip() or None
        elif key.strip() == 'budget' and info['budget'] is None:
            info['budget'] = parse_money(val)
        elif ('box office' in key or ('gross' in key and 'budget' not in key)) and info['BoxOffice'] is None:
            info['BoxOffice'] = parse_money(val)
    return info


# --- חילוץ עלילת הסרט מהדף ---

def _extract_plot(soup):
    # אסטרטגיה 1: HTML חדש של ויקיפדיה — כותרת h2/h3 בתוך div.mw-heading
    for div in soup.find_all('div', class_=re.compile(r'mw-heading')):
        h = div.find(re.compile(r'^h[23]$'))
        if h and re.match(r'^(Plot|Synopsis|Story)$', h.get_text(strip=True), re.I):
            paras = []
            for sib in div.find_next_siblings():
                if sib.name in ('h2', 'h3') or (
                        sib.name == 'div' and 'mw-heading' in ' '.join(sib.get('class', []))):
                    break
                if sib.name == 'p' and sib.get_text(strip=True):
                    paras.append(sib.get_text(' ', strip=True))
                if len(paras) == 3:
                    break
            if paras:
                return ' '.join(paras)

    # אסטרטגיה 1ב: HTML ישן של ויקיפדיה — <span id="Plot"> בתוך <h2>
    span = soup.find('span', id=re.compile(r'^(Plot|Synopsis|Story)$', re.I))
    if span:
        h_tag = span.find_parent(re.compile(r'^h[23]$'))
        if h_tag:
            start = (h_tag.parent
                     if 'mw-heading' in ' '.join(h_tag.parent.get('class', []))
                     else h_tag)
            paras = []
            for sib in start.find_next_siblings():
                if sib.name in ('h2', 'h3') or (
                        sib.name == 'div' and 'mw-heading' in ' '.join(sib.get('class', []))):
                    break
                if sib.name == 'p' and sib.get_text(strip=True):
                    paras.append(sib.get_text(' ', strip=True))
                if len(paras) == 3:
                    break
            if paras:
                return ' '.join(paras)

    # אסטרטגיה 2: חילוץ מהפסקאות הפותחות של הדף (כשאין קטע Plot נפרד)
    content = soup.find('div', id='mw-content-text')
    if content:
        target = content.find('div', class_='mw-parser-output') or content
        paras = []
        for elem in target.children:
            tag = getattr(elem, 'name', None)
            if tag in ('h2', 'h3', 'h4'):
                break
            if tag == 'div' and 'mw-heading' in ' '.join(elem.get('class', [])):
                break
            if tag == 'p':
                txt = elem.get_text(' ', strip=True)
                if txt:
                    paras.append(txt)
            if len(paras) == 2:
                break
        if paras:
            return ' '.join(paras)
    return None


# --- חיפוש ויקיפדיה כגיבוי כשכל כתובות ה-URL נכשלו ---

def _find_page_via_search(title, year_s):
    query = (title + ' ' + year_s + ' film').strip()
    url   = WIKI_BASE + '/w/index.php?search=' + requests.utils.quote(query) + '&fulltext=1'
    html  = _fetch(url)
    if not html:
        return None
    soup = BeautifulSoup(html, 'html.parser')
    first_word = re.split(r'\W+', title.lower())[0]
    # בדיקת 8 תוצאות ראשונות — מחפשים התאמה לפי מילה ראשונה ומאפייני סרט
    for item in soup.select('li.mw-search-result')[:8]:
        heading = item.select_one('.mw-search-result-heading a')
        if not heading:
            continue
        href = heading.get('href', '')
        if first_word not in href.lower().replace('_', ' '):
            continue
        desc = item.select_one('.searchresult')
        desc_txt = desc.get_text(' ', strip=True).lower() if desc else ''
        if 'film' not in desc_txt and 'movie' not in desc_txt and (not year_s or year_s not in desc_txt):
            continue
        page_url  = WIKI_BASE + href if href.startswith('/') else href
        page_html = _fetch(page_url)
        if not page_html:
            continue
        page_soup = BeautifulSoup(page_html, 'html.parser')
        ok, _ = _is_film_page(page_soup, year_s)
        if ok:
            return page_url, page_soup
    return None


# --- פונקציה ראשית לשליפת נתוני ויקיפדיה — בטוחה לריצה מ-threads מרובים ---

EMPTY = {'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None, 'plot': None}

def get_wiki_info(tconst, title, year):
    # בדיקת cache תחת מנעול לפני כל פנייה לרשת
    with cache_lock:
        if tconst in cache:
            return tconst, cache[tconst], False
        if tconst in miss_cache:
            # tconst זה אושר בעבר כלא קיים בויקיפדיה — מדלגים מיד
            return tconst, EMPTY, False

    did_fetch = False  # האם בוצעה בקשת HTTP בפועל

    try:
        year_s = str(int(year)) if pd.notna(year) else ''

        # רשימת כתובות URL לניסיון — מהמדויקת לכללית
        candidates = []
        if year_s:
            candidates.append(_title_to_url(f'{title} ({year_s} film)'))
        candidates.append(_title_to_url(title + ' (film)'))
        candidates.append(_title_to_url(title))

        soup = None
        for url in candidates:
            html = _fetch(url)
            if html is None:
                continue  # 404 — ממשיך לכתובת הבאה
            did_fetch = True

            s = BeautifulSoup(html, 'html.parser')

            # טיפול בעמוד disambiguation — מנסה לזהות את הסרט המתאים
            if _is_disambiguation(s):
                follow_url = _follow_disambiguation(s, year_s)
                if follow_url:
                    fhtml = _fetch(follow_url)
                    if fhtml:
                        fs = BeautifulSoup(fhtml, 'html.parser')
                        ok, _ = _is_film_page(fs, year_s)
                        if ok:
                            soup = fs
                            break
                continue

            ok, _ = _is_film_page(s, year_s)
            if ok:
                soup = s
                break

        # גיבוי: חיפוש ויקיפדיה אם כל הכתובות הישירות נכשלו
        if soup is None and did_fetch:
            found = _find_page_via_search(title, year_s)
            if found:
                _, soup = found

        if soup is None:
            with cache_lock:
                if not did_fetch:
                    # כל הכתובות החזירו 404 — מוסיפים למiss-cache לדילוג מהיר בעתיד
                    miss_cache.add(tconst)
            return tconst, EMPTY, did_fetch

        # חילוץ הנתונים מה-infobox ומקטע העלילה
        infobox = soup.find('table', class_=re.compile(r'infobox'))
        info = _extract_from_infobox(infobox) if infobox else {
            'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None}
        info['plot'] = _extract_plot(soup)

    except Exception:
        # שגיאת רשת זמנית — לא שומרים בcache, יחזור בריצה הבאה
        return tconst, EMPTY, did_fetch

    # שמירה בcache תחת מנעול
    with cache_lock:
        cache[tconst] = info

    return tconst, info, did_fetch


print('פונקציות עזר ל-Wikipedia scraping מוכנות.')
print(f'קבצי cache: {WIKI_CACHE_FILE} / {WIKI_MISS_FILE}')
print(f'Hit-cache (עם נתונים)   : {len(cache)}')
print(f'Miss-cache (לא נמצאו)   : {len(miss_cache)}')
remaining = len(movies) - len(cache) - len(miss_cache)
print(f'עדיין לשליפה            : ~{max(0, remaining)}')

## 5b. הרצת ה-scraping עם ThreadPoolExecutor

In [ ]:
# פרמטרי הריצה המקבילית
MAX_WORKERS  = 5      # מספר threads שרצים בו-זמנית (5 = מהיר אך נימוסי)
SAVE_EVERY   = 200    # שמירת cache לדיסק כל כמה סרטים שהסתיימו
FETCH_DELAY  = 0.2    # שניות המתנה אחרי כל בקשת HTTP אמיתית (לא בcache hits)

# בניית מילון תוצאות — מאכלס מראש בכל מה שכבר בcache
results = {}
with cache_lock:
    for tc in cache:
        results[tc] = cache[tc]

# רשימת הסרטים שעדיין צריך לשלוף (לא בcache ולא במiss-cache)
to_fetch = [
    row for row in movies.itertuples(index=False)
    if row.tconst not in cache and row.tconst not in miss_cache
]
print(f'כבר בcache: {len(results)} | לשליפה: {len(to_fetch)} | threads: {MAX_WORKERS}')

done_count  = 0
fetch_count = 0
save_lock   = threading.Lock()  # מגן על מונים וכתיבה לדיסק

def _worker(row):
    # פונקציית עבודה של כל thread — שולפת סרט אחד
    tconst, info, did_fetch = get_wiki_info(row.tconst, row.primaryTitle, row.startYear)
    if did_fetch:
        time.sleep(FETCH_DELAY)  # השהיה רק כשהייתה בקשת HTTP אמיתית
    return tconst, info, did_fetch

# הרצת כל הסרטים בthread pool — as_completed מחזיר כל future כשמסיים
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_tconst = {executor.submit(_worker, row): row.tconst for row in to_fetch}

    for future in as_completed(future_to_tconst):
        tconst, info, did_fetch = future.result()
        results[tconst] = info

        with save_lock:
            done_count += 1
            if did_fetch:
                fetch_count += 1

            # שמירה תקופתית של שני קבצי ה-cache לדיסק
            if done_count % SAVE_EVERY == 0:
                found = sum(1 for v in results.values() if any(x is not None for x in v.values()))
                print(f'  בוצעו: {done_count}/{len(to_fetch)} | '
                      f'נשלפו: {fetch_count} | נמצאו: {found} | '
                      f'miss-cache: {len(miss_cache)}')
                with open(WIKI_CACHE_FILE, 'w', encoding='utf-8') as f:
                    json.dump(cache, f, ensure_ascii=False)
                with open(WIKI_MISS_FILE, 'w', encoding='utf-8') as f:
                    json.dump(list(miss_cache), f)

# שמירה סופית של שני הcache לדיסק
with open(WIKI_CACHE_FILE, 'w', encoding='utf-8') as f:
    json.dump(cache, f, ensure_ascii=False)
with open(WIKI_MISS_FILE, 'w', encoding='utf-8') as f:
    json.dump(list(miss_cache), f)

# בניית DataFrame תוצאות בסדר המקורי של הסרטים
wiki_rows = [results.get(tc, EMPTY) for tc in movies['tconst']]
wiki_df = pd.DataFrame(wiki_rows, index=movies.index)

total = len(movies)
print()
print('Wikipedia scraping הסתיים.')
found_any = sum(1 for r in wiki_rows if any(v is not None for v in r.values()))
print(f'סרטים עם נתוני ויקיפדיה: {found_any}/{total}  ({100*found_any/total:.1f}%)')
print(f'גודל miss-cache (דילוג בריצה הבאה): {len(miss_cache)}')
print()
print('כיסוי לפי שדה:')
for col in ['Language', 'Country', 'budget', 'BoxOffice', 'plot']:
    n = wiki_df[col].notna().sum()
    print(f'  {col:<15}: {n:5d}  ({100*n/total:.1f}%)')

## 6. מיזוג ושמירת dataset_threaded.csv

In [ ]:
# הסרת עמודות ויקיפדיה ישנות אם קיימות (מניעת כפילויות בריצות חוזרות)
_WIKI_COLS = ['Language', 'Country', 'budget', 'BoxOffice', 'plot']
movies = movies.drop(columns=[c for c in _WIKI_COLS if c in movies.columns], errors='ignore')

# חיבור עמודות ויקיפדיה לטבלת הסרטים הראשית
movies = pd.concat([movies.reset_index(drop=True), wiki_df.reset_index(drop=True)], axis=1)

# סדר עמודות סופי לפי דרישות המטלה
cols = [
    'tconst', 'primaryTitle', 'startYear', 'genres', 'lead_actors_ids',
    'runtimeMinutes', 'averageRating', 'Language', 'Country',
    'numVotes', 'budget', 'BoxOffice', 'plot'
]
movies.drop(columns=['titleType'], errors='ignore', inplace=True)
movies = movies[cols]

total = len(movies)
print(f'גודל סט הנתונים: {total} סרטים')
print()
print('כיסוי שדות (% ערכים לא-null):')
cov = (movies.notna().mean() * 100).round(1)
for col, pct in cov.items():
    print(f'  {col:<20}: {pct}%')

In [ ]:
# המרת עמודות רשימה למחרוזת לפני שמירה ל-CSV
save = movies.copy()
save['genres']          = save['genres'].apply(str)
save['lead_actors_ids'] = save['lead_actors_ids'].apply(str)

# שמירת סט הנתונים הסופי
save.to_csv('dataset.csv', index=False, encoding='utf-8')

print('נשמר: dataset.csv')
print('שורות:', len(save), '  עמודות:', len(save.columns))
pd.read_csv('dataset.csv', nrows=3)